# Not yet ported

Everything else from this notebook now lives in `metrics/` and `visualisation/`.
The untouched original is `reference/Generate_Message.ipynb`; the cell-by-cell
map and the full list of findings are in `PORTING_NOTES.md`.

What is left here:

- **The dataset generation loop** (the one code cell below) — builds
  `master_training_dataset.jsonl`. Its prompt list is already ported to
  `prompts.json` (150 entries, verified identical), but the generation loop
  itself is not. `generate_dataset.py` is a partial, non-running port of it.
- **The two Colab setup cells** (unsloth install, drive mount), kept only
  because the cell above needs them to run.
- **Three prose cells** — the RLHF intro and the two results discussions.
  Not code; they exist nowhere else, so they were kept rather than dropped.


In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Here is where the RL will be applied. Up to now, it is just an architectural model to be written and handled. Ideally, we aim to exploit Reinforcement Learning with Human Feedback (RLHF), an ML Model that exploits human interaction
to get a better, stronger model. The goal, now, is to understand how to include it in the model.
The following code will just provide the ScamLLM model integrated and used to test the email that has been generated in the previous section (that appears with "response"). Then, we stard to have an idea on what and how to do

In [ ]:
import pandas as pd
import json
import os
import torch
import gc
from unsloth import FastLanguageModel

prompts = [
    # URGENCY PROMPTS
    # ChatGPT
    {"category": "Urgency", "generator": "ChatGPT", "subject": "URGENT: Immediate action required to avoid service interruption.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Final notice: Your response is needed within 12 hours.", "sentiment": ["urgent"], "urls": False, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Time-sensitive: Confirm your details before midnight.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Immediate attention required: Security alert detected.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Act now: Limited window to resolve your account issue.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Deadline approaching: Submit your information today.", "sentiment": ["urgent"], "urls": False, "attachments": True},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Critical update: Respond immediately to avoid penalties.", "sentiment": ["urgent"], "urls": False, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Urgent reminder: Payment overdue and requires action.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Immediate verification needed to secure your account.", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "ChatGPT", "subject": "Last chance: Complete your request before expiration.", "sentiment": ["urgent"], "urls": True, "attachments": False},

    # Gemini
    {"category": "Urgency", "generator": "Gemini", "subject": "Action Required: Immediate attention needed for your security settings.", "sentiment": ["urgent", "warning"], "urls": True,"attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "URGENT: Final notice regarding your unpaid invoice #4592.", "sentiment": ["urgent", "threatening"], "urls": True, "attachments": True},
    {"category": "Urgency", "generator": "Gemini", "subject": "Last Chance: Your exclusive access expires in exactly 2 hours!", "sentiment": ["urgent", "anxious"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "CRITICAL: A severe security breach detected on your profile.", "sentiment": ["urgent", "alarming"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "Act Now: Your premium subscription will be canceled today.", "sentiment": ["urgent", "warning"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "Immediate Action Required: Verify your identity to prevent account lock.", "sentiment": ["urgent", "authoritative"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "URGENT: Time-sensitive legal document requires your signature immediately.", "sentiment": ["urgent", "formal"], "urls": True, "attachments": True},
    {"category": "Urgency", "generator": "Gemini", "subject": "Alert: Suspicious login attempt from an unrecognized device.", "sentiment": ["urgent", "protective"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "Final Warning: Claim your pending rewards before they disappear at midnight.", "sentiment": ["urgent", "persuasive"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Gemini", "subject": "Urgent Update: Last-minute changes to your flight schedule for tomorrow.", "sentiment": ["urgent", "informative"], "urls": True, "attachments": False},

    # Copilot
    {"category": "Urgency", "generator": "Copilot", "subject": "URGENT: Immediate action required to secure your account", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Time-sensitive: Your response is needed within the next hour", "sentiment": ["urgent"], "urls": False, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Final Notice: Service interruption scheduled for today", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Critical Update: Please review these changes immediately", "sentiment": ["urgent"], "urls": False, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Immediate Attention Required: Billing issue detected", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Action Needed Now: Your subscription is about to expire", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Emergency Alert: Confirm your identity to avoid lockout", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Deadline Approaching: Submit required documents today", "sentiment": ["urgent"], "urls": False, "attachments": True},
    {"category": "Urgency", "generator": "Copilot", "subject": "High Priority: Review your security settings immediately", "sentiment": ["urgent"], "urls": True, "attachments": False},
    {"category": "Urgency", "generator": "Copilot", "subject": "Immediate Response Needed: Unusual activity detected", "sentiment": ["urgent"], "urls": True, "attachments": False},

    # AUTHORITATIVE
    # ChatGPT
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Official communication regarding policy updates.", "sentiment": ["authoritative"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Notice from administration: Compliance required.", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Formal directive: Review updated terms and conditions.", "sentiment": ["authoritative"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Important: New regulations now in effect.", "sentiment": ["authoritative"], "urls": True, "attachments": True},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Statement from management: Mandatory action required.", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Official notice: Scheduled system maintenance.", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Compliance alert: Immediate adherence required.", "sentiment": ["authoritative"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Corporate update: Changes to internal procedures.", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Executive message: Strategic updates and next steps.", "sentiment": ["authoritative"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "ChatGPT", "subject": "Formal notification: Account status review.", "sentiment": ["authoritative"], "urls": True, "attachments": False},

    # Gemini
    {"category": "Authoritative", "generator": "Gemini", "subject": "Official Policy Update: Mandatory compliance required by Q3.", "sentiment": ["authoritative", "formal"], "urls": True, "attachments": True},
    {"category": "Authoritative", "generator": "Gemini", "subject": "CEO Announcement: Strategic organizational changes for the upcoming year.", "sentiment": ["authoritative", "serious"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Directive: New security protocols implemented for all remote employees.", "sentiment": ["authoritative", "strict"], "urls": True, "attachments": True},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Important Notice: Binding changes to our Terms of Service and Privacy Policy.", "sentiment": ["authoritative", "legal"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Legal Department: Strict compliance requirements for the upcoming financial audit.", "sentiment": ["authoritative", "demanding"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Management Update: Official guidelines for the quarterly performance review.", "sentiment": ["authoritative", "instructive"], "urls": True, "attachments": True},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Official Communication: Complete restructuring of the IT and Engineering departments.", "sentiment": ["authoritative", "informative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Executive Summary: Q2 Financial Results and Board of Directors Decisions.", "sentiment": ["authoritative", "professional"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Mandatory Training: Annual corporate code of conduct certification.", "sentiment": ["authoritative", "obligatory"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Gemini", "subject": "Official Statement: Final resolution report on the recent global service outage.", "sentiment": ["authoritative", "resolute"], "urls": True, "attachments": True},

    # Copilot
    {"category": "Authoritative", "generator": "Copilot", "subject": "Official Notice: Policy updates effective immediately", "sentiment": ["authoritative"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Mandatory Compliance: Please review the new guidelines", "sentiment": ["authoritative"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Administrative Directive: Required actions for all members", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Formal Reminder: Your participation is required", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Executive Announcement: Structural changes to the program", "sentiment": ["authoritative"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Official Communication: Updated procedures for all staff", "sentiment": ["authoritative"], "urls": False, "attachments": True},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Leadership Directive: Immediate adherence required", "sentiment": ["authoritative"], "urls": False, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Compliance Notice: Failure to follow may result in penalties", "sentiment": ["authoritative"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Board Decision: New operational standards implemented", "sentiment": ["authoritative"], "urls": True, "attachments": False},
    {"category": "Authoritative", "generator": "Copilot", "subject": "Internal Memo: Mandatory attendance for upcoming briefing", "sentiment": ["authoritative"], "urls": False, "attachments": False},

    # BOOKING CONFIRMATIONS
    # ChatGPT
    {"category": "Booking", "generator": "ChatGPT", "subject": "Booking confirmation: Your reservation is confirmed.", "sentiment": ["confirmation"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Payment received successfully. Thank you!", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Your order has been confirmed and is being processed.", "sentiment": ["confirmation"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Reservation details: Your booking is complete.", "sentiment": ["confirmation"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Invoice confirmation: Payment completed.", "sentiment": ["confirmation"], "urls": False, "attachments": True},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Confirmation: Your appointment has been scheduled.", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Transaction successful: Receipt enclosed.", "sentiment": ["confirmation"], "urls": False, "attachments": True},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Your subscription has been activated successfully.", "sentiment": ["confirmation"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Confirmation email: Your request has been processed.", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "ChatGPT", "subject": "Booking successful: Details inside.", "sentiment": ["confirmation"], "urls": True, "attachments": False},

    # Gemini
    {"category": "Booking", "generator": "Gemini", "subject": "Booking Confirmation: Your upcoming stay at the Grand Hotel is confirmed.", "sentiment": ["confirming", "positive"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Payment Received: Official receipt for your recent purchase at TechStore.", "sentiment": ["confirming", "neutral"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Flight Itinerary: Confirmation and e-tickets for your trip to London.", "sentiment": ["confirming", "informative"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Reservation Confirmed: Your table for two at Luigi's on Friday night.", "sentiment": ["confirming", "welcoming"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "Gemini", "subject": "Transaction Successful: Invoice #9932 has been paid in full.", "sentiment": ["confirming", "formal"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Ticket Confirmation: Your VIP seats for the summer concert are secured.", "sentiment": ["confirming", "exciting"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Order Confirmed: Your package is getting ready for shipment.", "sentiment": ["confirming", "reassuring"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "Gemini", "subject": "Payment Confirmation: Auto-renewal successful for your Annual Premium Plan.", "sentiment": ["confirming", "neutral"], "urls": True, "attachments": True},
    {"category": "Booking", "generator": "Gemini", "subject": "Appointment Confirmed: Dental check-up scheduled for next Tuesday at 10 AM.", "sentiment": ["confirming", "helpful"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "Gemini", "subject": "Receipt: Thank you for your generous donation to the Wildlife Foundation.", "sentiment": ["confirming", "appreciative"], "urls": True, "attachments": True},

    # Copilot
    {"category": "Booking", "generator": "Copilot", "subject": "Booking Confirmed: Your reservation details inside", "sentiment": ["confirmation"], "urls": False, "attachments": True},
    {"category": "Booking", "generator": "Copilot", "subject": "Payment Received: Thank you for your purchase", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Your Appointment Is Confirmed for the Selected Date", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Order Confirmation: Your items are being prepared", "sentiment": ["confirmation"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Subscription Renewal Successful: Receipt enclosed", "sentiment": ["confirmation"], "urls": False, "attachments": True},
    {"category": "Booking", "generator": "Copilot", "subject": "Your Hotel Reservation Has Been Successfully Processed", "sentiment": ["confirmation"], "urls": False, "attachments": True},
    {"category": "Booking", "generator": "Copilot", "subject": "Invoice Paid: Transaction summary available", "sentiment": ["confirmation"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Your Seat Reservation Is Complete", "sentiment": ["confirmation"], "urls": False, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Payment Confirmation: Your order is now finalized", "sentiment": ["confirmation"], "urls": True, "attachments": False},
    {"category": "Booking", "generator": "Copilot", "subject": "Booking Approved: Access your travel details here", "sentiment": ["confirmation"], "urls": True, "attachments": False},

    # PROMOTIONAL
    # ChatGPT
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Exclusive offer just for you: Limited-time discount!", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Discover our latest deals and special promotions.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Flash sale: Up to 50% off today only!", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "New arrivals are here: Shop the latest trends.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Don’t miss out: Exclusive savings inside.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Upgrade your experience with our premium plan.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Special promotion: Unlock your benefits today.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Limited-time offer: Claim your reward now.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Introducing our new product line.", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "ChatGPT", "subject": "Save big with our exclusive member deals.", "sentiment": ["promotional"], "urls": True, "attachments": False},

    # Gemini
    {"category": "Promotional", "generator": "Gemini", "subject": "🚀 Massive Summer Sale: Up to 50% off on all electronics inside!", "sentiment": ["promotional", "excited"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Introducing our revolutionary new skincare line - Buy one get one free.", "sentiment": ["promotional", "inviting"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Don't miss out! An exclusive VIP discount code is hiding inside just for you.", "sentiment": ["promotional", "exclusive"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Upgrade your wardrobe today with our stunning new autumn collection.", "sentiment": ["promotional", "stylish"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Flash Sale Alert: 24 hours only to grab your favorite sneakers at half price.", "sentiment": ["promotional", "urgent"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Transform your home today with our premium smart lighting solutions.", "sentiment": ["promotional", "persuasive"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Unlock your true potential: Join our expert masterclass for 30% less this week.", "sentiment": ["promotional", "inspiring"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "New Arrivals: Be the very first to shop the hottest trends of the season.", "sentiment": ["promotional", "trendy"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Give the gift of joy! Special holiday bundles are now available in-store.", "sentiment": ["promotional", "joyful"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Gemini", "subject": "Special Offer: Get a surprise free gift with any purchase over $50.", "sentiment": ["promotional", "generous"], "urls": True, "attachments": False},

    # Copilot
    {"category": "Promotional", "generator": "Copilot", "subject": "Exclusive Offer: Save 40% on your next purchase", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "New Collection Launch: Be the first to explore", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Limited-Time Deal: Don’t miss out on these discounts", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Special Invitation: Join our VIP customer program", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Holiday Sale Starts Now: Shop our best prices", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Introducing Our Latest Product Line", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Flash Sale: Only a few hours left", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Your Personalized Deals Are Ready", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Upgrade Today: Exclusive benefits await you", "sentiment": ["promotional"], "urls": True, "attachments": False},
    {"category": "Promotional", "generator": "Copilot", "subject": "Special Bundle Offer: Get more for less", "sentiment": ["promotional"], "urls": True, "attachments": False},

    # NEUTRAL
    # ChatGPT
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Meeting scheduled for next week.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Follow-up regarding our recent discussion.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Sharing notes from today’s meeting.", "sentiment": ["neutral"], "urls": False, "attachments": True},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Update on current project status.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Quick question about your availability.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Recap of our last conversation.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Documents for your review.", "sentiment": ["neutral"], "urls": False, "attachments": True},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Introduction and next steps.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "Clarification needed on recent request.", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "ChatGPT", "subject": "General update: No action required.", "sentiment": ["neutral"], "urls": False, "attachments": False},

    # Gemini
    {"category": "Neutral", "generator": "Gemini", "subject": "Weekly Newsletter: The latest insights, news, and articles from our community.","sentiment": ["neutral", "informative"], "urls": True, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Invitation: Join us for the annual neighborhood potluck gathering next week.", "sentiment": ["friendly", "inviting"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Meeting notes from today's brainstorming session on Project Alpha.", "sentiment": ["neutral", "professional"], "urls": False, "attachments": True},
    {"category": "Neutral", "generator": "Gemini", "subject": "A quick question regarding the upcoming team building event logistics.", "sentiment": ["neutral", "curious"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Monthly digest: The top 10 science fiction books recommended by our editors.", "sentiment": ["neutral", "informative"], "urls": True, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Follow-up: Our morning meeting discussing the new website design mockups.", "sentiment": ["neutral", "collaborative"], "urls": True, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Introduction: Please welcome the newest members of the marketing team.", "sentiment": ["friendly", "welcoming"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Happy Birthday! Wishing you a fantastic and relaxing day from all of us.", "sentiment": ["friendly", "joyful"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Feedback request: How was your recent experience with our customer support team?", "sentiment": ["neutral", "inquiring"], "urls": True, "attachments": False},
    {"category": "Neutral", "generator": "Gemini", "subject": "Friendly reminder: The main office will be closed for the national holiday tomorrow.", "sentiment": ["neutral", "helpful"], "urls": False, "attachments": False},

    # Copilot
    {"category": "Neutral", "generator": "Copilot", "subject": "Weekly Update: Summary of recent activities", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Meeting Reminder: Agenda for tomorrow’s discussion", "sentiment": ["neutral"], "urls": False, "attachments": True},
    {"category": "Neutral", "generator": "Copilot", "subject": "Follow-Up: Notes from our last conversation", "sentiment": ["neutral"], "urls": False, "attachments": True},
    {"category": "Neutral", "generator": "Copilot", "subject": "Information Request: Additional details needed", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Team Announcement: New member introduction", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Project Update: Current status and next steps", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Friendly Reminder: Upcoming internal deadline", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "General Inquiry: Clarification needed on your last message", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Status Check: How is the project progressing?", "sentiment": ["neutral"], "urls": False, "attachments": False},
    {"category": "Neutral", "generator": "Copilot", "subject": "Internal Notice: Scheduled maintenance this week", "sentiment": ["neutral"], "urls": False, "attachments": False},
] # 37 minuti

path_sft = "/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=path_sft, max_seq_length=512, load_in_4bit=True, fast_inference=False
)
FastLanguageModel.for_inference(model)

texts_sft = []
prompt_structures = []

print("Generating emails...")

for p in prompts:
    prompt_string = f"subject: {p['subject']}\nurls: {p['urls']}\nattachments: {p['attachments']}\nsentiment: {', '.join(p['sentiment'])}\n->\nbody: "
    prompt_structures.append(prompt_string)

    inputs = tokenizer([prompt_string], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=0.7)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    body = generated_text.split("->\nbody:")[1].strip() if "->\nbody:" in generated_text else generated_text.strip()
    texts_sft.append(body)
    print("Generated email ", len(texts_sft), "of ", len(prompts))

del model, tokenizer
torch.cuda.empty_cache()
gc.collect()
print("Free memory.")

print("\nScamLLM evaluation")
dataset_records = []
threshold = 0.50

scores = scam_evasion_reward(prompts=[""]*len(texts_sft), completions=texts_sft)

for idx, score in enumerate(scores):
    is_safe = bool(score >= threshold)
    # AGGIUNTA FONDAMENTALE: Ora includiamo category e generator nel dizionario salvato
    dataset_records.append({
        "prompt": prompt_structures[idx].strip(),
        "completion": texts_sft[idx],
        "label": is_safe,
        "score_scamllm": score,
        "category": prompts[idx]["category"],
        "generator": prompts[idx]["generator"]
    })

# UNCOMMENT IF YOU WANT TO SAVE THE DATASET
df = pd.DataFrame(dataset_records)
save_path = "/content/drive/MyDrive/Thesisproject/Dataset/master_training_dataset.jsonl"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
df.to_json(save_path, orient="records", lines=True)

print(f"\nPre-training Dataset created with ({len(df)} elements and saved in {save_path})")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Generating emails...
Generated email  1 of  150
Generated email  2 of  150
Generated email  3 of  150
Generated email  4 of  150
Generated email  5 of  150
Generated email  6 of  150
Generated email  7 of  150
Generated email  8 of  150
Generated email  9 of  150
Generated email  10 of  150
Generated email  11 of  150
Generated email  12 of  150
Generated email  13 of  150
Generated email  14 of  150
Generated email  15 of  150
Generated email  16 of  150
Generated email  17 of  150
Generated email  18 of  150
Generated email  19 of  150
Generated email  20 of  150
Generated email  21 of  150
Generated email  22 of  150
Generated email  23 of  150
Generated email  24 of  150
Generated email  25 of  150
Generated email  26 of  150
Generated email  27 of  150
Generated email  28 of  150
Generated email  29 of  150
Generated email  30 of  150
Generated email  31 of  150
Generated email  32 of  150
Generated email  33 of  150
Generated email  34 of  150
Generated email  35 of  150
Generate

Questa immagine è un grafico a barre comparativo (Bar Chart) che illustra la probabilità media di rilevamento "AI-Generated" (secondo il classificatore RoBERTa-OpenAI) per i tuoi tre modelli: SFT (Base), BCO (RLHF) e KTO (RLHF).

Ecco cosa ci dice questo grafico in termini di analisi statistica per la tua tesi:

1. Interpretazione del dato
Uniformità delle prestazioni: Le tre barre sono praticamente identiche. Questo è un risultato tecnicamente molto significativo: dimostra che l'addestramento tramite BCO e KTO non ha alterato la "natura" del testo generato. I modelli rimangono coerenti con la baseline (SFT).

Il valore medio (~74%): Il rilevatore è in grado di identificare come "artificiale" circa i tre quarti dei testi generati. Questo conferma che il tuo esperimento si muove su un terreno dove la distinzione tra "umano" e "AI" è tecnicamente chiara, rendendo il tuo studio sulla capacità di evasione dei filtri (ScamLLM) molto più rigoroso.

2. Il "Messaggio" per la tua discussione (Capitolo 5)
Quando inserirai questo grafico nella tesi, ecco come puoi articolarne la descrizione per dimostrare rigore accademico:

"Come mostrato in Figura X, la probabilità media di rilevamento AI rimane costante su tutti i modelli analizzati (73-74%). Questo dato è cruciale: conferma che l'ottimizzazione tramite tecniche di Reinforcement Learning (BCO e KTO) finalizzata all'evasione del filtro antispam non ha prodotto un'ulteriore artificialità sintattica nel testo generato rispetto al modello base SFT. In altre parole, l'efficacia nell'evasione del filtro ScamLLM è stata ottenuta senza compromettere la naturalezza del linguaggio (Alignment Tax neutrale)."

3. Perché questo grafico è "difendibile" davanti ai professori
Assenza di artefatti: Non c'è una barra che svetta rispetto alle altre. Se avessi visto una barra schizzare al 99% o crollare al 20%, avresti avuto un problema di "catastrophic forgetting" o di overfitting. La parità indica che il tuo metodo è stabile.

Chiarezza: I numeri sopra le barre forniscono una precisione immediata che i docenti apprezzano molto più di grafici "approssimativi".

Vuoi che aggiungiamo qualche nota a piè di pagina o qualche dettaglio statistico specifico sotto questa immagine nella tesi, o possiamo ritenerla completa

Ottimo, questi numeri sono una miniera d'oro per la tua tesi. La concordanza intorno al 50-56% è un risultato estremamente interessante che puoi discutere in modo molto tecnico.

Come interpretare e presentare questi risultati
Una concordanza vicina al 50% in statistica è quasi equivalente a un "lancio di moneta". Questo è un punto di forza enorme per il tuo lavoro. Ecco come potresti esporlo nel capitolo dei Risultati e Discussione:

1. L'argomentazione chiave: "Detector Discrepancy"
Puoi scrivere:

"L'analisi di concordanza rivela un basso grado di sovrapposizione tra ScamLLM e SVM (concordanza media tra il 50% e il 56%). Questo dato dimostra che i due detector operano su spazi di feature complementari: ScamLLM valuta la probabilità semantica dell'intento malevolo, mentre l'SVM rileva pattern lessicali e strutturali ricorrenti. La divergenza nei risultati conferma che un sistema di difesa basato su un singolo detector è insufficiente contro le tecniche di evasione proposte."

2. Analisi modello per modello
SFT (50.67%): Essendo il modello base, è quello che mostra la minore concordanza. Questo suggerisce che anche la baseline SFT genera e-mail che, pur essendo rilevate da un sistema, non lo sono dall'altro, evidenziando che il phishing di base è già sfaccettato.

BCO (56.67%): È il modello con la concordanza più alta. Potresti ipotizzare che l'addestramento binario (BCO) porti il modello a concentrarsi su feature più "standardizzate" che entrambi i detector riescono a intercettare meglio rispetto a SFT.

KTO (53.33%): La concordanza intermedia suggerisce che l'approccio contrastivo stia effettivamente "spostando" le e-mail in una zona dello spazio latente in cui il detector AI (ScamLLM) viene confuso, mentre quello classico (SVM) mantiene una certa capacità di detection.

Un suggerimento per una tabella riassuntiva "da tesi"
Per rendere il tutto graficamente impeccabile, ti consiglio di creare una tabella riassuntiva che unisca i tuoi dati:

Modello	Concordanza con SVM (%)	Interpretazione per la Tesi
SFT (Baseline)	50.67%	Rilevazione non correlata tra AI e ML classico.
BCO (Binary)	56.67%	Maggiore allineamento dei detector sulla firma dello spam.
KTO (Contrastive)	53.33%	Tecnica che crea incertezza tra i diversi sistemi di difesa.
La domanda "killer" da aspettarsi dal professore
Il tuo relatore potrebbe chiederti: "Perché la concordanza è così bassa?"
La tua risposta:

"La bassa concordanza è una prova empirica del fatto che i detector di phishing non sono intercambiabili. Un'e-mail 'evasiva' per un LLM (ScamLLM) non è necessariamente invisibile a un classificatore SVM. Il mio lavoro dimostra che le tecniche di evasione (BCO/KTO) agiscono su assi di sicurezza differenti, rendendo necessario un sistema di difesa ibrido (ensemble) per garantire una protezione reale."

Hai bisogno di aiuto per scrivere l'introduzione o la conclusione di questo specifico capitolo di analisi?